# Bangla Wikipedia — Atomic Fact Extraction (Qwen3.5-9B, transformers + 4-bit, Kaggle T4x2)

This notebook runs **only the fact-extraction step** of the bucketing pipeline:

`raw paragraph -> list of atomic (subject, attribute, value) facts`

QA generation from buckets is a separate later notebook — not included here.

**Design choices for Kaggle T4x2:**
- Uses **transformers + bitsandbytes 4-bit quantization** with `device_map="auto"` (same approach as the working QA generation notebook — no vLLM compatibility issues).
- Writes results to `PROGRESS_PATH` as **JSONL, one line per row, appended incrementally** — this is what makes the notebook resumable if a Kaggle session times out or crashes mid-run.
- Every `CHECKPOINT_EVERY` rows, progress is flushed to disk.
- On restart, already-processed IDs are loaded from `PROGRESS_PATH` and skipped automatically — just re-run the notebook.
- A separate `FAILED_PATH` logs rows where the model's output couldn't be parsed as JSON, so you can inspect/retry them separately instead of silently losing them.


## 1. Config

Update `DATA_DIR` / `INPUT_PATH` to match your Kaggle dataset mount path.

In [ ]:
DATA_DIR = "/kaggle/input/your-dataset-slug"   # <-- update this
INPUT_PATH = f"{DATA_DIR}/wikipedia.csv"        # <-- update filename if different

OUTPUT_PATH = "/kaggle/working/facts.json"
PROGRESS_PATH = "/kaggle/working/facts_progress.jsonl"
FAILED_PATH = "/kaggle/working/failed_rows.jsonl"

CHECKPOINT_EVERY = 20     # rows per checkpoint to disk
BATCH_SIZE = 8             # contexts per model.generate() call — raise if VRAM allows

MODEL_NAME = "Qwen/Qwen3.5-9B"
MAX_NEW_TOKENS = 800
TEMPERATURE = 0.2   # low temp: extraction should be consistent, not creative


## 2. Install dependencies

Uses the same stack as the working QA generation notebook: `transformers` + `accelerate` + `bitsandbytes`.  
No vLLM — avoids Kaggle compatibility issues.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

## 3. Load data

In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_PATH, usecols=["id", "title", "text", "url"])
df["text"] = df["text"].fillna("").astype(str)
df["title"] = df["title"].fillna("").astype(str)

print(f"Loaded {len(df)} rows.")
df.head()


## 4. Fact-extraction prompt

Only the extraction step — no QA generation here.


In [ ]:
FACT_EXTRACTION_PROMPT = """তুমি একজন তথ্য নিষ্কাশন বিশেষজ্ঞ। নিচের বাংলা প্যারাগ্রাফ থেকে প্রতিটি স্বতন্ত্র, পরমাণু-পর্যায়ের তথ্য (subject, attribute, value) triple আকারে বের করো।

নিয়মাবলি:
1. প্রতিটি তারিখ/সাল, সংখ্যা, নাম, স্থান, উপাধি, পদবি, প্রতিষ্ঠানের নাম আলাদা triple হিসেবে দেখাও।
2. একটি বাক্যে একাধিক তথ্য থাকলে প্রতিটি আলাদাভাবে বের করো।
3. subject অবশ্যই স্পষ্ট হতে হবে (সর্বনাম নয়, প্রকৃত নাম/সত্তা)। যদি প্যারাগ্রাফের মূল বিষয় (যেমন ব্যক্তি/স্থান/প্রতিষ্ঠানের নাম) অস্পষ্টভাবে "তিনি/এরা/এটি" দিয়ে উল্লেখ থাকে, তাহলে প্রকৃত নাম বসাও (যদি প্যারাগ্রাফে আগে উল্লেখ থাকে)।
4. প্যারাগ্রাফে উল্লেখ নেই এমন কোনো তথ্য নিজে থেকে যোগ করবে না।
5. যদি প্যারাগ্রাফে অর্থবহ কোনো তথ্য না থাকে, খালি array ফেরত দাও।
6. আউটপুট শুধুমাত্র বৈধ JSON array হবে, অন্য কোনো ব্যাখ্যা, ভূমিকা বা মার্কডাউন কোড ফেন্স থাকবে না।

ফরম্যাট:
[{{"subject": "...", "attribute": "...", "value": "..."}}, ...]

প্যারাগ্রাফ:
{paragraph}
"""

print(FACT_EXTRACTION_PROMPT.format(paragraph="(example paragraph goes here)")[:300], "...")


## 5. Load model (4-bit quantized) — same approach as working QA notebook

Uses `transformers` + `bitsandbytes` 4-bit quantization with `device_map="auto"`,  
identical to `bengali_qa_generation_qwen3.5.ipynb`. No vLLM dependency.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("GPUs available:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  {i}: {torch.cuda.get_device_name(i)} - {props.total_memory/1e9:.1f} GB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.eval()

print("Model loaded. Device map:", getattr(model, "hf_device_map", "n/a"))

## 6. Build prompts + batched generation helper

Port of the same `generate_batch` pattern from the working QA notebook.

In [ ]:
def build_prompt(paragraph: str):
    messages = [
        {"role": "user", "content": FACT_EXTRACTION_PROMPT.format(paragraph=paragraph)}
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


@torch.inference_mode()
def generate_batch(paragraphs):
    messages_list = [[{"role": "user", "content": FACT_EXTRACTION_PROMPT.format(paragraph=p)}] for p in paragraphs]

    try:
        prompts = [
            tokenizer.apply_chat_template(
                m, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            for m in messages_list
        ]
    except TypeError:
        prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in messages_list
        ]

    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    generated = output_ids[:, input_len:]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)

## 7. JSON parsing helper

Handles the common failure modes: stray ```json fences, leading/trailing text around the array.


In [ ]:
import re
import json

def parse_facts_json(raw_text: str):
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                return None
        return None


## 8. Resume support

Loads IDs already present in `PROGRESS_PATH` so re-running the notebook after an interruption
only processes what's left.


In [ ]:
import os

def load_processed_ids(progress_path: str) -> set:
    processed = set()
    if os.path.exists(progress_path):
        with open(progress_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    processed.add(rec["id"])
                except json.JSONDecodeError:
                    continue
    return processed

processed_ids = load_processed_ids(PROGRESS_PATH)
print(f"Already processed: {len(processed_ids)} rows.")

df_remaining = df[~df["id"].isin(processed_ids)].reset_index(drop=True)
print(f"Remaining: {len(df_remaining)} / {len(df)} rows.")


## 9. Main extraction loop

Processes `BATCH_SIZE` rows at a time through batched `model.generate()`, then
appends results to `PROGRESS_PATH` and flushes to disk every `CHECKPOINT_EVERY` rows.
Safe to interrupt — just re-run the notebook from the top to resume.


In [ ]:
import time

progress_f = open(PROGRESS_PATH, "a", encoding="utf-8")
failed_f = open(FAILED_PATH, "a", encoding="utf-8")

total = len(df_remaining)
start_time = time.time()
rows_since_checkpoint = 0

for start in range(0, total, BATCH_SIZE):
    chunk = df_remaining.iloc[start:start + BATCH_SIZE]
    paragraphs = chunk["text"].tolist()

    raw_outputs = generate_batch(paragraphs)

    for row, raw_text in zip(chunk.itertuples(index=False), raw_outputs):
        facts = parse_facts_json(raw_text)

        record = {
            "id": row.id,
            "title": row.title,
            "text": row.text,
            "facts": facts if facts is not None else [],
        }
        progress_f.write(json.dumps(record, ensure_ascii=False) + "\n")

        if facts is None:
            failed_f.write(json.dumps(
                {"id": row.id, "title": row.title, "raw_output": raw_text},
                ensure_ascii=False,
            ) + "\n")

    rows_since_checkpoint += len(chunk)

    if rows_since_checkpoint >= CHECKPOINT_EVERY:
        progress_f.flush()
        failed_f.flush()
        rows_since_checkpoint = 0

    done = start + len(chunk)
    elapsed = time.time() - start_time
    rate = done / elapsed if elapsed > 0 else 0
    print(f"[{done}/{total}] processed "
          f"({len(processed_ids) + done}/{len(df)} total) "
          f"— {rate:.2f} rows/sec")

progress_f.flush()
progress_f.close()
failed_f.flush()
failed_f.close()
print("Done.")


## 10. Consolidate into final `facts.json`

Reads the JSONL progress file and writes a single JSON array. Deduplicates by `id`
(last write wins) in case of any overlapping reruns.


In [ ]:
records_by_id = {}

with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        records_by_id[rec["id"]] = rec

records = list(records_by_id.values())

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Saved {len(records)} records to {OUTPUT_PATH}")

# quick sanity check
n_empty = sum(1 for r in records if len(r["facts"]) == 0)
n_failed = sum(1 for _ in open(FAILED_PATH, encoding="utf-8")) if os.path.exists(FAILED_PATH) else 0
print(f"Rows with zero facts extracted: {n_empty}")
print(f"Rows that failed JSON parsing (see {FAILED_PATH}): {n_failed}")


## Notes

- **Resuming after a Kaggle session restart:** re-mount the same dataset, re-run cells 1–8 (this rebuilds `processed_ids` from `PROGRESS_PATH`, which persists in `/kaggle/working/` across a session but is lost if you start a *brand new* session — download/save `facts_progress.jsonl` as a Kaggle Dataset or output if the CSV is large enough that this will span multiple sessions).
- **Tuning throughput:** `BATCH_SIZE` controls how many paragraphs go through `model.generate()` per GPU call. Start with 8, raise to 16 if VRAM allows. `CHECKPOINT_EVERY` only affects disk flush frequency.
- **`MAX_NEW_TOKENS`:** Wikipedia paragraphs are mostly short, but a few (like the মোস্তফা কামাল example) are long multi-section articles. If you see truncated JSON in `FAILED_PATH` for long rows, raise `MAX_NEW_TOKENS`.
- **Why not vLLM?** Kaggle's environment has compatibility issues with vLLM on T4 GPUs. The `transformers` + `bitsandbytes` approach is proven to work (same stack as the QA generation notebook).
- **Next step (not in this notebook):** bucket `facts.json` into groups of ~4–6 facts per row and run the QA-generation prompt per bucket, as discussed earlier in this pipeline.
